# S&P 500 Stock Market ML Pipeline


```bash
pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib
```

## 1. Imports & Config

In [1]:
import warnings
from pathlib import Path

import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, f1_score,
    precision_recall_curve, precision_score, recall_score,
    roc_auc_score, roc_curve,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

In [2]:
RANDOM_STATE = 42
PROJECT_DIR = Path.cwd()
RAW_CSV = PROJECT_DIR / "sp500_stocks.csv"
OUTPUT_CSV = PROJECT_DIR / "sp500_enriched.csv"
MODEL_DIR = PROJECT_DIR / "models"
IMAGE_DIR = PROJECT_DIR / "images"

MODEL_DIR.mkdir(exist_ok=True)
IMAGE_DIR.mkdir(exist_ok=True)

## 2. Feature Engineering Functions

In [3]:
def compute_rsi(series, period=14):
    delta = series.diff()
    gain = delta.clip(lower=0)
    loss = -delta.clip(upper=0)
    avg_gain = gain.ewm(alpha=1.0/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1.0/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100.0 - (100.0 / (1.0 + rs))


def compute_macd(series, fast=12, slow=26, signal=9):
    ema_fast = series.ewm(span=fast, adjust=False).mean()
    ema_slow = series.ewm(span=slow, adjust=False).mean()
    macd_line = ema_fast - ema_slow
    signal_line = macd_line.ewm(span=signal, adjust=False).mean()
    return macd_line, signal_line


def compute_bollinger(series, period=20, num_std=2.0):
    sma = series.rolling(window=period).mean()
    std = series.rolling(window=period).std()
    upper = sma + num_std * std
    lower = sma - num_std * std
    return upper, lower

In [4]:
def enrich_single_ticker(df_ticker):
    df = df_ticker.sort_values("date").reset_index(drop=True)
    close, high, low, opn = df["close"], df["high"], df["low"], df["open"]
    volume = df["volume"].astype(float)

    df["daily_return"] = close.pct_change()
    df["log_return"] = np.log(close / close.shift(1))

    df["sma_20"] = close.rolling(20).mean()
    df["sma_50"] = close.rolling(50).mean()
    df["ema_12"] = close.ewm(span=12, adjust=False).mean()
    df["ema_26"] = close.ewm(span=26, adjust=False).mean()
    df["rsi_14"] = compute_rsi(close, 14)
    macd_line, signal_line = compute_macd(close)
    df["macd"] = macd_line
    df["macd_signal"] = signal_line
    bb_upper, bb_lower = compute_bollinger(close)
    df["bb_upper"] = bb_upper
    df["bb_lower"] = bb_lower

    log_ret = np.log(close / close.shift(1))
    df["rolling_vol_5"] = log_ret.rolling(5).std() * np.sqrt(252)
    df["rolling_vol_10"] = log_ret.rolling(10).std() * np.sqrt(252)

    df["volume_pct_change"] = volume.pct_change() * 100
    df["high_low_range_pct"] = ((high - low) / close) * 100
    df["open_close_change_pct"] = ((close - opn) / opn) * 100

    df["momentum_3"] = close / close.shift(3) - 1
    df["momentum_5"] = close / close.shift(5) - 1

    df["lag_close_1"] = close.shift(1)
    daily_ret = close.pct_change()
    df["lag_return_1"] = daily_ret.shift(1)
    df["lag_return_3"] = daily_ret.shift(3)
    df["lag_return_5"] = daily_ret.shift(5)

    df["target"] = (close.shift(-1) > close).astype(int)

    df = df.dropna().reset_index(drop=True)
    for col in df.select_dtypes(include=[np.floating]).columns:
        df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df = df.dropna().reset_index(drop=True)
    return df

## 3. Build Enriched Dataset

In [5]:
print(f"Loading {RAW_CSV} ...")
df_raw = pd.read_csv(RAW_CSV)
df_raw["date"] = pd.to_datetime(df_raw["date"])
print(f"Raw shape: {df_raw.shape}")
df_raw.head()

Loading C:\Users\Atharv\OneDrive\Desktop\SP500 ML Stock Prediction\sp500_stocks.csv ...
Raw shape: (16130, 8)


,date,close,high,low,open,volume,ticker,sector
0,2020-01-02,72.333893,72.394101,71.091199,71.344069,135480400,AAPL,Technology
1,2020-01-03,71.630653,72.389273,71.406681,71.563221,146322800,AAPL,Technology
2,2020-01-06,72.201439,72.239973,70.503576,70.754043,118387200,AAPL,Technology
3,2020-01-07,71.861847,72.466330,71.642689,72.211049,108872000,AAPL,Technology
4,2020-01-08,73.017822,73.318862,71.565606,71.565606,132079200,AAPL,Technology


In [6]:
frames = []
for ticker in df_raw["ticker"].unique():
    df_t = df_raw[df_raw["ticker"] == ticker].copy()
    df_t = enrich_single_ticker(df_t)
    frames.append(df_t)
    print(f"  {ticker}: {len(df_t)} rows")

df = pd.concat(frames, ignore_index=True)
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)
df.to_csv(OUTPUT_CSV, index=False)
print(f"\nEnriched shape: {df.shape}")
df.head()

  AAPL: 1564 rows
  AMZN: 1564 rows
  GOOGL: 1564 rows
  HD: 1564 rows
  JNJ: 1564 rows
  JPM: 1564 rows
  MSFT: 1564 rows
  NVDA: 1564 rows
  PG: 1564 rows
  XOM: 1564 rows

Enriched shape: (15640, 31)


,date,close,high,low,open,volume,ticker,sector,daily_return,log_return,...,volume_pct_change,high_low_range_pct,open_close_change_pct,momentum_3,momentum_5,lag_close_1,lag_return_1,lag_return_3,lag_return_5,target
0,2020-03-13,67.102943,67.573683,61.063026,63.945388,370732000,AAPL,Technology,0.119808,0.113158,...,-11.408594,9.702492,4.937893,-0.025828,-0.038266,59.923595,-0.098754,0.072021,-0.013280,0
1,2020-03-16,58.470348,62.542821,57.936845,58.407581,322423600,AAPL,Technology,-0.128647,-0.137708,...,-13.030545,7.877456,0.107464,-0.120611,-0.090018,67.102943,0.119808,-0.034730,-0.079092,1
2,2020-03-17,61.041286,62.187949,57.550590,59.749776,324056000,AAPL,Technology,0.043970,0.043031,...,0.506290,7.597086,2.161531,0.018652,-0.113829,58.470348,-0.128647,-0.098754,0.072021,0
3,2020-03-18,59.546997,60.350871,57.241593,57.881315,300233600,AAPL,Technology,-0.024480,-0.024785,...,-7.351322,5.221553,2.877755,-0.112602,-0.104418,61.041286,0.043970,0.119808,-0.034730,0
4,2020-03-19,59.090744,61.036455,58.566899,59.720807,271857200,AAPL,Technology,-0.007662,-0.007692,...,-9.451440,4.179261,-1.055015,0.010610,-0.013899,59.546997,-0.024480,-0.128647,-0.098754,0


## 4. Preprocessing

In [7]:
CATEGORICAL_COLS = ["ticker", "sector"]
BASE_COLS = ["open", "high", "low", "close", "volume"]
ENGINEERED_COLS = [
    "daily_return", "log_return",
    "sma_20", "sma_50", "ema_12", "ema_26",
    "rsi_14", "macd", "macd_signal",
    "bb_upper", "bb_lower",
    "rolling_vol_5", "rolling_vol_10",
    "volume_pct_change",
    "high_low_range_pct", "open_close_change_pct",
    "momentum_3", "momentum_5",
    "lag_close_1", "lag_return_1", "lag_return_3", "lag_return_5",
]
TARGET = "target"

cat_cols = [c for c in CATEGORICAL_COLS if c in df.columns]
num_cols = [c for c in BASE_COLS + ENGINEERED_COLS if c in df.columns]
feature_cols = num_cols + cat_cols

X = df[feature_cols]
y = df[TARGET]

split_idx = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

preprocessor = ColumnTransformer([
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_cols),
    ("num", StandardScaler(), num_cols),
], remainder="drop")

X_train_np = preprocessor.fit_transform(X_train)
X_test_np = preprocessor.transform(X_test)

ohe_names = list(preprocessor.named_transformers_["cat"].get_feature_names_out(cat_cols))
feature_names = ohe_names + num_cols

y_train_np = y_train.values.astype(int)
y_test_np = y_test.values.astype(int)

print(f"Train: {X_train_np.shape[0]} samples | Test: {X_test_np.shape[0]} samples")
print(f"Features: {X_train_np.shape[1]}")

Train: 12512 samples | Test: 3128 samples
Features: 41


## 5. Training

In [8]:
models = {
    "logistic_regression": LogisticRegression(
        max_iter=2000, random_state=RANDOM_STATE, C=1.0, solver="lbfgs",
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=200, max_depth=12, min_samples_split=10,
        min_samples_leaf=5, random_state=RANDOM_STATE, n_jobs=-1,
    ),
    "xgboost": XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE,
        eval_metric="logloss", n_jobs=-1,
    ),
}

trained = {}
for name, model in models.items():
    model.fit(X_train_np, y_train_np)
    tr_acc = model.score(X_train_np, y_train_np)
    te_acc = model.score(X_test_np, y_test_np)
    trained[name] = model
    print(f"  {name:25s} | Train Acc: {tr_acc:.4f} | Test Acc: {te_acc:.4f}")

  logistic_regression       | Train Acc: 0.5322 | Test Acc: 0.5288
  random_forest             | Train Acc: 0.7810 | Test Acc: 0.5352
  xgboost                   | Train Acc: 0.9270 | Test Acc: 0.5064


## 6. Evaluation

In [9]:
DISPLAY_NAMES = {
    "logistic_regression": "Logistic Regression",
    "random_forest": "Random Forest",
    "xgboost": "XGBoost",
}
COLORS = {
    "logistic_regression": "#3b82f6",
    "random_forest": "#10b981",
    "xgboost": "#f59e0b",
}

predictions, probabilities, rows = {}, {}, []

for name, model in trained.items():
    y_pred = model.predict(X_test_np)
    y_prob = model.predict_proba(X_test_np)[:, 1]
    predictions[name] = y_pred
    probabilities[name] = y_prob
    rows.append({
        "Model": DISPLAY_NAMES[name],
        "Accuracy": accuracy_score(y_test_np, y_pred),
        "Precision": precision_score(y_test_np, y_pred, zero_division=0),
        "Recall": recall_score(y_test_np, y_pred, zero_division=0),
        "F1 Score": f1_score(y_test_np, y_pred, zero_division=0),
        "ROC AUC": roc_auc_score(y_test_np, y_prob),
    })

comparison = pd.DataFrame(rows).sort_values("F1 Score", ascending=False).reset_index(drop=True)
comparison

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.528772,0.529528,0.970962,0.685312,0.509394
1,Random Forest,0.535166,0.534609,0.929825,0.678887,0.517878
2,XGBoost,0.506394,0.527484,0.632789,0.575358,0.494962


In [10]:
for name in trained:
    print(f"\n--- {DISPLAY_NAMES[name]} ---")
    print(classification_report(y_test_np, predictions[name], target_names=["DOWN", "UP"]))


--- Logistic Regression ---
              precision    recall  f1-score   support

        DOWN       0.51      0.03      0.06      1475
          UP       0.53      0.97      0.69      1653

    accuracy                           0.53      3128
   macro avg       0.52      0.50      0.37      3128
weighted avg       0.52      0.53      0.39      3128


--- Random Forest ---
              precision    recall  f1-score   support

        DOWN       0.54      0.09      0.16      1475
          UP       0.53      0.93      0.68      1653

    accuracy                           0.54      3128
   macro avg       0.54      0.51      0.42      3128
weighted avg       0.54      0.54      0.43      3128


--- XGBoost ---
              precision    recall  f1-score   support

        DOWN       0.47      0.36      0.41      1475
          UP       0.53      0.63      0.58      1653

    accuracy                           0.51      3128
   macro avg       0.50      0.50      0.49      3128
weigh

In [11]:
best = comparison.loc[0, "Model"]
f1_val = comparison.loc[0, "F1 Score"]
print(f"Best model: {best} (F1={f1_val:.4f})")

comparison.to_csv(PROJECT_DIR / "model_comparison.csv", index=False)
for name, model in trained.items():
    joblib.dump(model, MODEL_DIR / f"{name}.joblib")
joblib.dump(preprocessor, MODEL_DIR / "preprocessor.joblib")
joblib.dump(feature_names, MODEL_DIR / "feature_names.joblib")
print(f"Models saved to {MODEL_DIR}/")

Best model: Logistic Regression (F1=0.6853)
Models saved to C:\Users\Atharv\OneDrive\Desktop\SP500 ML Stock Prediction\models/


## 7. Visualizations

In [12]:
def save_fig(name):
    plt.savefig(IMAGE_DIR / name, dpi=150, bbox_inches="tight", facecolor="white")
    plt.close()
    print(f"  Saved: images/{name}")

In [13]:
# Class Distribution
counts = pd.Series(y_test_np).value_counts().sort_index()
fig, ax = plt.subplots(figsize=(6, 5))
bars = ax.bar(["DOWN (0)", "UP (1)"], counts.values, color=["#ef4444", "#10b981"],
              edgecolor="white", linewidth=1.5, width=0.5)
for bar, c in zip(bars, counts.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + counts.max()*0.02,
            str(c), ha="center", va="bottom", fontsize=12, fontweight="bold")
ax.set_ylabel("Count")
ax.set_title("Target Class Distribution", fontsize=13, fontweight="bold")
ax.set_ylim(0, counts.max() * 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig("class_distribution.png")
plt.show()

  Saved: images/class_distribution.png


In [14]:
# Correlation Heatmap
num_cols_corr = [c for c in feature_names if c in df.columns]
if len(num_cols_corr) >= 2:
    corr = df[num_cols_corr].corr()
    fig, ax = plt.subplots(figsize=(14, 12))
    mask = np.triu(np.ones_like(corr, dtype=bool))
    sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0,
                ax=ax, square=True, linewidths=0.3, cbar_kws={"shrink": 0.8})
    ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
    plt.tight_layout()
    save_fig("correlation_heatmap.png")
    plt.show()

  Saved: images/correlation_heatmap.png


In [15]:
# Feature Importance
tree_models = {k: v for k, v in trained.items() if k in ["random_forest", "xgboost"]}
fig, axes = plt.subplots(1, len(tree_models), figsize=(8*len(tree_models), 8))
if len(tree_models) == 1:
    axes = [axes]
for ax, (name, model) in zip(axes, tree_models.items()):
    imp = model.feature_importances_
    n = min(len(imp), len(feature_names))
    imp_df = pd.DataFrame({"Feature": feature_names[:n], "Importance": imp[:n]})
    imp_df = imp_df.sort_values("Importance", ascending=True).tail(20)
    ax.barh(imp_df["Feature"], imp_df["Importance"], color=COLORS[name], edgecolor="white")
    ax.set_title(DISPLAY_NAMES[name], fontsize=12, fontweight="bold")
    ax.set_xlabel("Importance")
plt.suptitle("Feature Importance (Top 20)", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
save_fig("feature_importance.png")
plt.show()

  Saved: images/feature_importance.png


In [16]:
# ROC Curve
fig, ax = plt.subplots(figsize=(8, 6))
for name, y_prob in probabilities.items():
    fpr, tpr, _ = roc_curve(y_test_np, y_prob)
    auc_val = roc_auc_score(y_test_np, y_prob)
    ax.plot(fpr, tpr, color=COLORS[name], linewidth=2,
            label=f"{DISPLAY_NAMES[name]} (AUC={auc_val:.4f})")
ax.plot([0, 1], [0, 1], "k--", linewidth=1, alpha=0.5, label="Random (AUC=0.50)")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate")
ax.set_title("ROC Curve", fontsize=13, fontweight="bold")
ax.legend(loc="lower right")
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
save_fig("roc_curve.png")
plt.show()

  Saved: images/roc_curve.png


In [17]:
# Precision-Recall Curve
fig, ax = plt.subplots(figsize=(8, 6))
for name, y_prob in probabilities.items():
    prec, rec, _ = precision_recall_curve(y_test_np, y_prob)
    ax.plot(rec, prec, color=COLORS[name], linewidth=2, label=DISPLAY_NAMES[name])
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve", fontsize=13, fontweight="bold")
ax.legend(loc="lower left")
ax.set_xlim([-0.02, 1.02])
ax.set_ylim([-0.02, 1.02])
plt.tight_layout()
save_fig("precision_recall_curve.png")
plt.show()

  Saved: images/precision_recall_curve.png


In [18]:
# Confusion Matrices
fig, axes = plt.subplots(1, len(predictions), figsize=(6*len(predictions), 5))
if len(predictions) == 1:
    axes = [axes]
for ax, (name, y_pred) in zip(axes, predictions.items()):
    cm = confusion_matrix(y_test_np, y_pred)
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax,
                xticklabels=["DOWN", "UP"], yticklabels=["DOWN", "UP"],
                cbar=False, linewidths=0.5, linecolor="gray")
    ax.set_title(DISPLAY_NAMES[name], fontsize=12, fontweight="bold")
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Actual")
plt.suptitle("Confusion Matrices", fontsize=14, fontweight="bold", y=1.02)
plt.tight_layout()
save_fig("confusion_matrices.png")
plt.show()

  Saved: images/confusion_matrices.png


In [19]:
# Model Comparison
fig, ax = plt.subplots(figsize=(10, 5))
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1 Score", "ROC AUC"]
x = np.arange(len(comparison))
width = 0.15
cmap = ["#3b82f6", "#10b981", "#f59e0b", "#ef4444", "#8b5cf6"]
for i, metric in enumerate(metrics_to_plot):
    vals = comparison[metric].values
    bars = ax.bar(x + i*width, vals, width, label=metric, color=cmap[i],
                  edgecolor="white", linewidth=0.5)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
                f"{val:.3f}", ha="center", va="bottom", fontsize=7, fontweight="bold")
ax.set_xlabel("Model")
ax.set_ylabel("Score")
ax.set_title("Model Comparison", fontsize=13, fontweight="bold")
ax.set_xticks(x + width*2)
ax.set_xticklabels(comparison["Model"].values, fontsize=10)
ax.legend(loc="lower right", fontsize=9, ncol=5)
ax.set_ylim(0, 1.15)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
plt.tight_layout()
save_fig("model_comparison.png")
plt.show()

  Saved: images/model_comparison.png


In [20]:
# Stock Technical Charts (first 3 tickers)
for ticker in df["ticker"].unique()[:3]:
    sub = df[df["ticker"] == ticker].sort_values("date").tail(200).copy()
    dates = sub["date"]

    fig, axes = plt.subplots(4, 1, figsize=(14, 14), sharex=True,
                             gridspec_kw={"height_ratios": [3, 1, 1, 1]})

    ax = axes[0]
    ax.plot(dates, sub["close"], color="#1a1a2e", linewidth=1.5, label="Close")
    if "sma_20" in sub.columns:
        ax.plot(dates, sub["sma_20"], color="#3b82f6", linewidth=1, alpha=0.8, label="SMA 20")
    if "sma_50" in sub.columns:
        ax.plot(dates, sub["sma_50"], color="#ef4444", linewidth=1, alpha=0.8, label="SMA 50")
    ax.set_title(f"{ticker} - Price with SMA 20 & SMA 50", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylabel("Price")

    ax = axes[1]
    if "rsi_14" in sub.columns:
        ax.plot(dates, sub["rsi_14"], color="#8b5cf6", linewidth=1)
        ax.axhline(70, color="#ef4444", linestyle="--", linewidth=0.8, alpha=0.7)
        ax.axhline(30, color="#10b981", linestyle="--", linewidth=0.8, alpha=0.7)
        ax.fill_between(dates, 70, sub["rsi_14"].clip(upper=70), alpha=0.15, color="#ef4444")
        ax.fill_between(dates, 30, sub["rsi_14"].clip(lower=30), alpha=0.15, color="#10b981")
    ax.set_title("RSI (14)", fontsize=11, fontweight="bold")
    ax.set_ylim(0, 100)
    ax.set_ylabel("RSI")

    ax = axes[2]
    if "macd" in sub.columns and "macd_signal" in sub.columns:
        ax.plot(dates, sub["macd"], color="#3b82f6", linewidth=1, label="MACD")
        ax.plot(dates, sub["macd_signal"], color="#ef4444", linewidth=1, label="Signal")
        hist = sub["macd"] - sub["macd_signal"]
        cols = ["#10b981" if v >= 0 else "#ef4444" for v in hist]
        ax.bar(dates, hist, color=cols, alpha=0.6, width=1.5)
    ax.set_title("MACD", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylabel("MACD")

    ax = axes[3]
    if "bb_upper" in sub.columns and "bb_lower" in sub.columns:
        ax.plot(dates, sub["close"], color="#1a1a2e", linewidth=1.2, label="Close")
        ax.plot(dates, sub["bb_upper"], color="#f59e0b", linewidth=0.8, linestyle="--", label="BB Upper")
        ax.plot(dates, sub["bb_lower"], color="#f59e0b", linewidth=0.8, linestyle="--", label="BB Lower")
        ax.fill_between(dates, sub["bb_upper"], sub["bb_lower"], alpha=0.1, color="#f59e0b")
    ax.set_title("Bollinger Bands", fontsize=11, fontweight="bold")
    ax.legend(fontsize=9)
    ax.set_ylabel("Price")

    plt.tight_layout()
    safe = ticker.replace(".", "_")
    save_fig(f"stock_chart_{safe}.png")
    plt.show()

  Saved: images/stock_chart_AAPL.png
  Saved: images/stock_chart_AMZN.png
  Saved: images/stock_chart_GOOGL.png
